# 05 — Full Research Summary

**SGPO v2: Early-Stage CKD Detection Using Hybrid Co-Evolutionary Optimization**

**Team:** Muhammed Abdel-Hamid | Abdul Rahman Al-Bili

**Supervisor:** Dr. Ibrahim | New Mansoura University | CSE015

---

This notebook consolidates all research results into a single presentation-ready overview.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'font.family': 'serif'})
plt.style.use('seaborn-v0_8-whitegrid')

RESULTS = '../results'

with open(f'{RESULTS}/baseline_rf_results.json') as f: baseline = json.load(f)
with open(f'{RESULTS}/sgpo_v2_results.json') as f: sgpo = json.load(f)
with open(f'{RESULTS}/model_comparison_results.json') as f: mc = json.load(f)
with open(f'{RESULTS}/ablation_results.json') as f: abl = json.load(f)
conv = pd.read_csv(f'{RESULTS}/convergence_history.csv')

print('All results loaded.')

## 1. Dataset Overview

| Property | Value |
|---|---|
| Source | MIMIC-IV (PhysioNet) |
| Patients | 57,875 |
| CKD cases | 29,233 (50.5%) |
| Non-CKD controls | 28,642 (49.5%) |
| Features | 42 (demographics, admissions, insurance, marital, labs) |
| CKD definition | ICD-10 N18* / ICD-9 585* |

## 2. SGPO v2 Framework

Three co-evolutionary optimizers running simultaneously per generation:

| Component | Algorithm | Role |
|---|---|---|
| Feature Selection | SFOA (Starfish Optimization) | Binary mask optimization |
| HP Tuning | DOA (Dream Optimization) | Continuous parameter search |
| Noise Handling | Fungal Growth Optimizer | Perturbation injection |

**Fitness:** `0.50 * AUC - 0.20 * (features/total) + 0.30 * Sensitivity`

**Validation:** 10-fold outer CV (unseen) + 3-fold inner CV (optimization)

## 3. Main Result: Baseline vs SGPO v2

In [ ]:
opt = sgpo['optimization_results']
final = sgpo['final_evaluation']

comparison = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Sensitivity', 'Accuracy', 'Features'],
    'Baseline': [baseline['cv_auc_mean'], baseline['cv_sensitivity_mean'], baseline['cv_accuracy_mean'], 42],
    'SGPO v2': [final['auc_mean'], final['sensitivity_mean'], final['accuracy_mean'], opt['best_n_features']],
})
comparison['Change'] = comparison['SGPO v2'] - comparison['Baseline']
comparison['Change %'] = (comparison['Change'] / comparison['Baseline'] * 100).round(2)
print(comparison.to_string(index=False))

print(f"\nSelected features ({opt['best_n_features']}): {opt['selected_features']}")
print(f"Optimized HP: {opt['best_hyperparameters']}")

In [ ]:
# Visual comparison
metrics = ['AUC-ROC', 'Sensitivity', 'Accuracy']
b_vals = [baseline['cv_auc_mean'], baseline['cv_sensitivity_mean'], baseline['cv_accuracy_mean']]
s_vals = [final['auc_mean'], final['sensitivity_mean'], final['accuracy_mean']]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - 0.175, b_vals, 0.35, label='Baseline RF (42 feat.)', color='#6B7280', alpha=0.85)
ax.bar(x + 0.175, s_vals, 0.35, label='SGPO v2 (8 feat.)', color='#2563EB', alpha=0.85)
ax.set_ylim(0.85, 0.98)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_title('Baseline vs SGPO v2', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(conv['generation'], conv['best_fitness'], 'b-o', markersize=3, lw=2)
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Best Fitness')
axes[0].set_title('Fitness Convergence')

axes[1].plot(conv['generation'], conv['gen_n_features'], 'r-^', markersize=3, lw=2)
axes[1].fill_between(conv['generation'], conv['gen_n_features'], alpha=0.15, color='red')
axes[1].axhline(y=42, color='gray', ls='--', alpha=0.5, label='Baseline')
axes[1].set_xlabel('Generation')
axes[1].set_ylabel('Features')
axes[1].set_title('Feature Reduction')
axes[1].legend()

spore_gens = conv[conv['fgo_action'].str.contains('spore', na=False)]['generation']
for g in spore_gens:
    for ax in axes:
        ax.axvline(x=g, color='orange', ls=':', alpha=0.7)

plt.suptitle('SGPO v2 Optimization Progress', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Model Comparison

In [ ]:
mc_df = pd.DataFrame(mc['models'])
print(mc_df[['model', 'n_features', 'auc_mean', 'sensitivity_mean', 'f1_mean']].to_string(index=False))

## 6. Ablation Study

In [ ]:
abl_df = pd.DataFrame(abl['variants'])
print(abl_df[['variant', 'n_features', 'auc_mean', 'sensitivity_mean']].to_string(index=False))

## 7. Key Findings

1. **81% feature reduction** (42 to 8) with only 0.25% AUC loss
2. **Only 1 lab test** (creatinine) needed for CKD detection
3. **Co-evolution justified**: all three components contribute (ablation study)
4. **SGPO v2 competitive**: 99.4% of XGBoost AUC with 19% of features
5. **Clinically practical**: 8-feature model is interpretable and deployable

## 8. Remaining Work

- Write full research paper (Introduction, Methods, Results, Discussion)
- Prepare presentation slides
- Final submission to Dr. Ibrahim